# Data Statistics
This section provides a general overview of the statistical analysis performed on the dataset collected from the arXiv. The contents include the **number of papers**, **average paper size**, **overall success rate**, **charts**, and **citation counts**. The analysis is based on the data collected during the crawling and discovery process, as well as the reference harvesting from the Semantic Scholar API.

In [11]:
import json
import pandas as pd
import plotly.express as px
from itables import show
import itables.options as opt

opt.lengthMenu = [5, 10, 25, 50]

# Đọc dữ liệu từ thư mục data/milestone1/
file_path = "../data/milestone1/discovery_log.json" 

with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Đã tải thành công {len(df)} bản ghi.")
df.head()

Đã tải thành công 20000 bản ghi.


,arxiv_id,title,authors,submission_date,latest_updated,revised_dates,venue,latest_version,versions,abstract,status
0,2511.11709,Thermal properties of Klein-Gordon Oscillator ...,"[Abdelmalek Boumali, Nosratollah Jafari, Bekda...",2025-11-12T15:47:33Z,2025-11-12T15:47:33Z,[],"Eur.Phys.J.C 85 (2025) 10, 1147",1,[2511.11709v1],We examine the thermal and statistical propert...,found
1,2511.11710,Target-Balanced Score Distillation,"[Zhou Xu, Qi Wang, Yuxiao Yang, Luyuan Zhang, ...",2025-11-12T15:53:01Z,2025-11-12T15:53:01Z,[],None,1,[2511.11710v1],Score Distillation Sampling (SDS) enables 3D a...,found
2,2511.11711,Which Sparse Autoencoder Features Are Real? Mo...,[Tsogt-Ochir Enkhbayar],2025-11-12T17:12:45Z,2025-11-12T17:12:45Z,[],None,1,[2511.11711v1],Although sparse autoencoders (SAEs) are crucia...,found
3,2511.11712,Reasoning: From Reflection to Solution,[Zixi Li],2025-11-12T17:23:32Z,2025-11-12T17:23:32Z,[],None,1,[2511.11712v1],What is reasoning? This question has driven ce...,found
4,2511.11713,Understanding the Representation of Older Adul...,"[Yunkai Yu, Yingying Wang, Rong Zheng]",2025-11-12T18:04:13Z,2025-11-12T18:04:13Z,[],None,1,[2511.11713v1],The Internet of Things (IoT) sensors have been...,found


In [12]:
total_records = len(df)

# Tính Number of papers (số bài status = 'found')
if 'status' in df.columns:
    success_count = len(df[df['status'] == 'found'])
    overall_success_rate = (success_count / total_records) * 100 if total_records > 0 else 0
    
    print(f"📊 **SUMMARY METRICS:**")
    print(f"- Tổng số lượng bản ghi quét (Total Records): {total_records}")
    print(f"- Số bài thu thập thành công (Number of Papers - Found): {success_count}")
    print(f"- Tỷ lệ thành công tổng thể (Overall Success Rate): {overall_success_rate:.2f}%")
else:
    print(f"- Number of Papers: {total_records}")
    print(f"- Overall Success Rate: 100.0% (Không tìm thấy trường trạng thái status)")

📊 **SUMMARY METRICS:**
- Tổng số lượng bản ghi quét (Total Records): 20000
- Số bài thu thập thành công (Number of Papers - Found): 20000
- Tỷ lệ thành công tổng thể (Overall Success Rate): 100.00%


In [13]:
# Kiểm tra nếu có trường chứa dung lượng file (ví dụ: 'size_kb' hoặc 'file_size')
size_col = next((col for col in ['size_kb', 'file_size', 'size'] if col in df.columns), None)

if size_col:
    avg_size = df[size_col].mean()
    print(f"📦 **Average Paper Size:** {avg_size:.2f} KB")
else:
    print("ℹ️ Trường dung lượng bài báo (size) chưa có sẵn trực tiếp trong discovery_log.json.")
    print("   (Chỉ số này sẽ được tính toán khi tích hợp dữ liệu từ Bước 2 - Full-Text Source Download).")

ℹ️ Trường dung lượng bài báo (size) chưa có sẵn trực tiếp trong discovery_log.json.
   (Chỉ số này sẽ được tính toán khi tích hợp dữ liệu từ Bước 2 - Full-Text Source Download).


In [14]:
# 1. Biểu đồ phân bố theo thời gian nộp (Submission Date)
if 'submission_date' in df.columns:
    df['submission_date'] = pd.to_datetime(df['submission_date'], errors='coerce')
    df['year_month'] = df['submission_date'] .dt.to_period('M').astype(str)
    df_time = df.groupby('year_month').size().reset_index(name='paper_count')
    
    fig_time = px.bar(df_time, x='year_month', y='paper_count',
                      title='<b>Biểu đồ 1: Số lượng bài báo theo thời gian (Monthly Distribution)</b>',
                      labels={'year_month': 'Tháng/Năm', 'paper_count': 'Số lượng bài báo'},
                      template='plotly_white')
    fig_time.update_xaxes(tickangle=-45)
    fig_time.show()

# 2. Biểu đồ phân bố số lượng phiên bản (Versions / Revisions)
if 'versions' in df.columns:
    df['version_count'] = df['versions'].apply(lambda x: len(x) if isinstance(x, list) else 1)
    version_dist = df['version_count'].value_counts().sort_index().reset_index()
    version_dist.columns = ['Version_Count', 'Papers']
    version_dist['Version_Count'] = version_dist['Version_Count'].astype(str) + ' Version(s)'
    
    fig_ver = px.pie(version_dist, names='Version_Count', values='Papers',
                     title='<b>Biểu đồ 2: Tỷ lệ phân bố số lượng phiên bản chỉnh sửa (Revisions)</b>',
                     hole=0.4, template='plotly_white')
    fig_ver.update_traces(textposition='inside', textinfo='percent+label')
    fig_ver.show()

C:\Users\PC\AppData\Local\Temp\ipykernel_22508\3065894499.py:4: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.



In [15]:
# Kiểm tra nếu có trường citation_count hoặc references
citation_col = next((col for col in ['citation_count', 'citations'] if col in df.columns), None)

if citation_col:
    avg_citations = df[citation_col].mean()
    max_citations = df[citation_col].max()
    
    print(f"📈 **CITATION COUNTS STATISTICS:**")
    print(f"- Số lượt trích dẫn trung bình (Average Citations): {avg_citations:.2f}")
    print(f"- Số lượt trích dẫn cao nhất (Max Citations): {max_citations}")
    
    # Biểu đồ phân bố citation
    fig_cite = px.histogram(df, x=citation_col, nbins=30,
                           title='<b>Biểu đồ 3: Phân bố số lượng trích dẫn (Citation Counts Distribution)</b>',
                           labels={citation_col: 'Số lượng trích dẫn', 'count': 'Số lượng bài báo'},
                           template='plotly_white')
    fig_cite.show()
else:
    print("ℹ️ Dữ liệu Citation Counts chưa có trong discovery_log.json (Dữ liệu này thuộc về Milestone 3 - Semantic Scholar Reference Extraction).")
    print("   Khi có dữ liệu trích dẫn, bạn có thể đưa trực tiếp vào cột 'citation_count' để tự động sinh thống kê và biểu đồ.")

ℹ️ Dữ liệu Citation Counts chưa có trong discovery_log.json (Dữ liệu này thuộc về Milestone 3 - Semantic Scholar Reference Extraction).
   Khi có dữ liệu trích dẫn, bạn có thể đưa trực tiếp vào cột 'citation_count' để tự động sinh thống kê và biểu đồ.


In [16]:
available_cols = [c for c in ['arxiv_id', 'title', 'venue', 'submission_date', 'status'] if c in df.columns]

if available_cols:
    print("📋 **Bảng dữ liệu mẫu thu thập từ arXiv:**")
    show(df[available_cols].head(50))
else:
    show(df.head(50))

📋 **Bảng dữ liệu mẫu thu thập từ arXiv:**
